# EPI Recorder v2.8.9 - Google Colab Demo

This notebook shows the full EPI product loop in Colab:

- install `epi-recorder`
- create a policy
- record a good agent run
- record a failing agent run
- analyze policy violations
- verify trust state
- inspect the `.epi` contents
- extract and render the embedded viewer
- tamper with an artifact and prove trust drops to `NONE`

The main flow works without any external model key. An optional final section also demonstrates the new OpenAI Agents-style event bridge in `v2.8.9`.


## 1. Install EPI Recorder

This notebook is written for `epi-recorder==2.8.9`.


In [ ]:
%pip install -q epi-recorder==2.8.9

import epi_core
print('Installed epi-recorder', epi_core.__version__)


## 2. Set up the Colab workspace

We keep everything under `/content/epi_colab_demo` so the artifacts are easy to inspect and download.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK = Path('/content/epi_colab_demo')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

ARTIFACTS = WORK / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

print('Workspace:', WORK)
print('Artifacts:', ARTIFACTS)


## 3. Create a policy rulebook

This policy is intentionally business-shaped. It says:

- manager approval is required before refund approval
- only approved tools may be used
- refunds above the threshold need explicit approval quality


In [ ]:
policy = {
    'policy_format_version': '2.0',
    'policy_id': 'refund-demo-v2',
    'system_name': 'refund-agent-demo',
    'system_version': '2.8.9',
    'policy_version': '2026-03-24',
    'scope': {
        'environment': 'demo',
        'workflow': 'customer_refund',
        'team': 'support-ops'
    },
    'approval_policies': [
        {
            'approval_id': 'manager-refund-approval',
            'required_roles': ['manager'],
            'minimum_approvers': 1,
            'reason_required': True
        }
    ],
    'rules': [
        {
            'id': 'R005',
            'name': 'Manager Approval Before Refund',
            'severity': 'critical',
            'description': 'Refund approval requires an approved manager response.',
            'type': 'approval_guard',
            'mode': 'require_approval',
            'applies_at': ['decision'],
            'approval_action': 'approve_refund',
            'approval_policy_ref': 'manager-refund-approval'
        },
        {
            'id': 'R006',
            'name': 'Only Approved Refund Tools',
            'severity': 'high',
            'description': 'Only allowlisted tools may be called in this workflow.',
            'type': 'tool_permission_guard',
            'mode': 'detect',
            'applies_at': ['tool_call'],
            'allowed_tools': ['lookup_order', 'verify_identity', 'issue_refund', 'send_email']
        }
    ]
}

policy_path = WORK / 'epi_policy.json'
policy_path.write_text(json.dumps(policy, indent=2), encoding='utf-8')
print(policy_path.read_text())


## 4. Validate the policy with the CLI

This exercises the improved `epi policy validate` diagnostics shipped in `v2.8.9`.


In [ ]:
subprocess.run([sys.executable, '-m', 'epi_cli.main', 'policy', 'validate', 'epi_policy.json'], check=True)


## 5. Record a good agent run and a failing agent run

We use the Python API directly so this notebook works reliably in Colab.


In [ ]:
from epi_recorder import record


def make_good_run(path: Path):
    with record(path, workflow_name='Refund Agent Demo', goal='Resolve a customer refund safely') as epi:
        with epi.agent_run(
            'refund-agent',
            user_input='Refund order ORD-1001',
            session_id='sess-good-001',
            task_id='refund-ORD-1001',
            attempt=1,
        ) as agent:
            agent.plan('Look up the order, verify identity, request approval, then decide.')
            agent.message('user', 'Refund order ORD-1001')
            agent.memory_read('refund_history', query='ORD-1001', result_count=1)
            agent.tool_call('lookup_order', {'order_id': 'ORD-1001'})
            agent.tool_result('lookup_order', {'status': 'paid', 'amount': 120, 'customer_id': 'CUST-42'})
            agent.tool_call('verify_identity', {'customer_id': 'CUST-42'})
            agent.tool_result('verify_identity', {'verified': True})
            agent.approval_request('approve_refund', reason='Manual approval required for refund action')
            agent.approval_response('approve_refund', approved=True, reviewer='manager', notes='Approved after identity verification')
            agent.decision('approve_refund', confidence=0.94)
            agent.tool_call('issue_refund', {'order_id': 'ORD-1001', 'amount': 120})
            agent.tool_result('issue_refund', {'success': True, 'refund_id': 'RF-1001'})


def make_bad_run(path: Path):
    with record(path, workflow_name='Refund Agent Demo', goal='Resolve a customer refund safely') as epi:
        with epi.agent_run(
            'refund-agent',
            user_input='Refund order ORD-2002',
            session_id='sess-bad-001',
            task_id='refund-ORD-2002',
            attempt=1,
        ) as agent:
            agent.plan('Look up the order, then decide quickly.')
            agent.message('user', 'Refund order ORD-2002')
            agent.tool_call('lookup_order', {'order_id': 'ORD-2002'})
            agent.tool_result('lookup_order', {'status': 'paid', 'amount': 500, 'customer_id': 'CUST-77'})
            agent.tool_call('admin_override', {'order_id': 'ORD-2002'})
            agent.tool_result('admin_override', {'status': 'override used'})
            agent.decision('approve_refund', confidence=0.62)
            agent.tool_call('issue_refund', {'order_id': 'ORD-2002', 'amount': 500})
            agent.tool_result('issue_refund', {'success': True, 'refund_id': 'RF-2002'})


good_artifact = ARTIFACTS / 'refund_agent_ok.epi'
bad_artifact = ARTIFACTS / 'refund_agent_missing_approval.epi'

make_good_run(good_artifact)
make_bad_run(bad_artifact)

print('Created:', good_artifact)
print('Created:', bad_artifact)


## 6. Analyze and verify both artifacts

The good artifact should pass cleanly. The bad artifact should show policy failures.


In [ ]:
def run_cli(*args):
    print('$ epi', ' '.join(args))
    result = subprocess.run([sys.executable, '-m', 'epi_cli.main', *args], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print('exit_code =', result.returncode)
    print('-' * 80)
    return result


run_cli('analyze', str(good_artifact))
run_cli('verify', str(good_artifact))
run_cli('analyze', str(bad_artifact))
run_cli('verify', str(bad_artifact))


## 7. Inspect the sealed contents inside a `.epi` file

This shows why EPI is more than logs: policy, analysis, trust metadata, and the embedded viewer are all inside one artifact.


In [ ]:
def inspect_epi(path: Path):
    print('=' * 80)
    print(path.name)
    with zipfile.ZipFile(path, 'r') as zf:
        names = sorted(zf.namelist())
        for name in names:
            print(' -', name)
        manifest = json.loads(zf.read('manifest.json').decode('utf-8'))
        print('\nspec_version =', manifest.get('spec_version'))
        print('signature_present =', bool(manifest.get('signature')))
        if 'analysis.json' in names:
            analysis = json.loads(zf.read('analysis.json').decode('utf-8'))
            print('fault_detected =', analysis.get('fault_detected'))
            print('headline =', analysis.get('summary', {}).get('headline'))
        if 'policy_evaluation.json' in names:
            evaluation = json.loads(zf.read('policy_evaluation.json').decode('utf-8'))
            print('controls_evaluated =', evaluation.get('controls_evaluated'))
            print('controls_failed =', evaluation.get('controls_failed'))
            for item in evaluation.get('results', [])[:3]:
                print('  *', item.get('rule_id'), item.get('status'))


inspect_epi(good_artifact)
inspect_epi(bad_artifact)


## 8. Extract and render the embedded viewer

This uses `epi view --extract` and renders the generated `viewer.html` inside an iframe so Colab shows the real embedded artifact viewer.


In [ ]:
from IPython.display import HTML, display
import html as html_lib

viewer_dir = WORK / 'bad_viewer'
subprocess.run(
    [sys.executable, '-m', 'epi_cli.main', 'view', str(bad_artifact), '--extract', str(viewer_dir)],
    check=True,
)

viewer_html = (viewer_dir / 'viewer.html').read_text(encoding='utf-8')
iframe_srcdoc = html_lib.escape(viewer_html, quote=True)
display(HTML(f'<iframe srcdoc="{iframe_srcdoc}" style="width:100%;height:980px;border:1px solid #d1d5db;border-radius:18px;background:white;box-shadow:0 10px 30px rgba(0,0,0,0.06);"></iframe>'))


## 9. Tamper with an artifact and prove trust drops

This modifies `analysis.json` inside a copied artifact. The content may still open, but trust must drop to `NONE`.


In [ ]:
tampered_artifact = ARTIFACTS / 'refund_agent_missing_approval_tampered.epi'
shutil.copy2(bad_artifact, tampered_artifact)

with zipfile.ZipFile(tampered_artifact, 'a') as zf:
    analysis = json.loads(zf.read('analysis.json').decode('utf-8'))
    analysis['summary']['headline'] = 'FORGED ANALYSIS - DO NOT TRUST'
    zf.writestr('analysis.json', json.dumps(analysis, indent=2))

run_cli('verify', str(bad_artifact))
run_cli('verify', str(tampered_artifact))


## 10. Optional: demonstrate the OpenAI Agents-style event bridge

This does not require the OpenAI Agents SDK. We feed synthetic stream events into `OpenAIAgentsRecorder` so you can see the `v2.8.9` bridge surface in a notebook-friendly way.


In [ ]:
from epi_recorder.integrations import OpenAIAgentsRecorder

bridge_artifact = ARTIFACTS / 'openai_agents_bridge_demo.epi'
synthetic_events = [
    {'type': 'run_item_stream_event', 'item': {'type': 'message', 'role': 'assistant', 'content': 'I will reset the password safely.'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'tool_call', 'tool_name': 'lookup_user', 'arguments': {'email': 'customer@example.com'}, 'call_id': 'call-1'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'tool_result', 'tool_name': 'lookup_user', 'result': {'user_id': 'user-123'}, 'call_id': 'call-1'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'approval_request', 'action': 'reset_password', 'reason': 'High-risk account action'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'approval_response', 'action': 'reset_password', 'approved': True, 'reviewer': 'manager', 'notes': 'Approved'}},
    {'type': 'run_item_stream_event', 'item': {'type': 'final_output', 'output': 'Password reset completed successfully.'}},
]

with record(bridge_artifact, workflow_name='OpenAI Agents Bridge Demo') as epi:
    with OpenAIAgentsRecorder(epi, agent_name='support-agent', user_input='Reset customer password') as recorder:
        for event in synthetic_events:
            recorder.consume(event)

inspect_epi(bridge_artifact)


## 11. Download artifacts from Colab

Use this if you want to pull the generated `.epi` files onto your machine.


In [ ]:
from google.colab import files

print('Available artifacts:')
for path in sorted(ARTIFACTS.glob('*.epi')):
    print(' -', path.name)

# Uncomment one of these to download a file:
# files.download(str(good_artifact))
# files.download(str(bad_artifact))
# files.download(str(tampered_artifact))
# files.download(str(bridge_artifact))
